In [2]:
# Download the dataset using wget
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# Load the dataset
df = pd.read_csv('course_lead_scoring_2026.csv')

# --- Data Preparation ---
# Identify categorical and numerical columns
categorical_cols = list(df.select_dtypes(include=['object', 'str']).columns)
numerical_cols = list(df.select_dtypes(exclude=['object']).columns)

# Remove the target variable 'converted' from numerical features
if 'converted' in numerical_cols:
    numerical_cols.remove('converted')

# Replace missing values
df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# ==========================================
# Question 1
# ==========================================
q1_mode = df['industry'].mode()[0]
print(f"Question 1 Answer: {q1_mode}")

# ==========================================
# Question 2
# ==========================================
# Calculate correlation matrix for numerical features
corr_matrix = df[numerical_cols].corr().abs()
# Unstack, drop self-correlations (which are 1.0), and sort
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs < 1.0].sort_values(ascending=False)
highest_corr_pair = corr_pairs.index[0]

print(f"Question 2 Answer: {highest_corr_pair[0]} and {highest_corr_pair[1]}")

# ==========================================
# Split the data
# ==========================================
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Extract target variables
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Remove target from dataframes
del df_train['converted']
del df_val['converted']
del df_test['converted']

# ==========================================
# Question 3
# ==========================================
def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[categorical_cols].apply(calculate_mi)
mi_scores = mi_scores.sort_values(ascending=False)

print(f"Question 3 Answer: {mi_scores.index[0]} (Score: {round(mi_scores.iloc[0], 2)})")

# ==========================================
# Question 4
# ==========================================
dv = DictVectorizer(sparse=False)

# Dict transformation for One-Hot Encoding
train_dict = df_train[categorical_cols + numerical_cols].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_cols + numerical_cols].to_dict(orient='records')
X_val = dv.transform(val_dict)

# Train model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Calculate accuracy
y_pred = model.predict(X_val)
acc_original = accuracy_score(y_val, y_pred)

print(f"Question 4 Answer: {round(acc_original, 2)}")

# ==========================================
# Question 5
# ==========================================
features = categorical_cols + numerical_cols
diffs = {}

# Exclude each feature, train, and evaluate
for f in features:
    subset = features.copy()
    subset.remove(f)
    
    dv_subset = DictVectorizer(sparse=False)
    train_dict_subset = df_train[subset].to_dict(orient='records')
    X_train_subset = dv_subset.fit_transform(train_dict_subset)
    
    val_dict_subset = df_val[subset].to_dict(orient='records')
    X_val_subset = dv_subset.transform(val_dict_subset)
    
    model_subset = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_subset.fit(X_train_subset, y_train)
    
    y_pred_subset = model_subset.predict(X_val_subset)
    acc_subset = accuracy_score(y_val, y_pred_subset)
    
    diffs[f] = acc_original - acc_subset

# Filter differences for the options provided in the homework
options_q5 = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
smallest_diff_feature = min(options_q5, key=lambda x: diffs[x])

print(f"Question 5 Answer: {smallest_diff_feature}")

# ==========================================
# Question 6
# ==========================================
c_values = [0.000001, 0.00001, 0.0001, 0.001]
best_c = None
best_acc = -1

# Train using all features as in Q4
for c in c_values:
    model_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    y_pred_c = model_c.predict(X_val)
    acc_c_rounded = round(accuracy_score(y_val, y_pred_c), 3)
    
    # Track the best C (if tie, select smallest C)
    if acc_c_rounded > best_acc:
        best_acc = acc_c_rounded
        best_c = c
    elif acc_c_rounded == best_acc:
        if best_c is None or c < best_c:
            best_c = c

print(f"Question 6 Answer: {best_c}")

--2026-10-19 18:48:52--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.1’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.05s   

2026-10-19 18:48:52 (5.11 MB/s) - ‘course_lead_scoring_2026.csv.1’ saved [278746/278746]

Question 1 Answer: technology
Question 2 Answer: interaction_count and lead_score
Question 3 Answer: lead_source (Score: 0.03)
Question 4 Answer: 0.65
Question 5 Answer: number_of_courses_viewed
Question 6 Answer: 0.001
